We'll use the embedder.py script from the embed/ directory for generating embeddings.

Under the hood, it does four things:

Tokenize - convert text into integer IDs and attention masks

Run ONNX model - execute the model graph on CPU

Mean pooling - average the token embeddings, weighted by the attention mask

Normalize - divide by L2 norm so vectors can be compared with dot product

In [1]:
from embed.embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [2]:
v1.dot(dv)
v2.dot(dv)

np.float64(0.01973045838992233)

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from ingest import load_faq_data
from tqdm.auto import tqdm
import numpy as np

# Load the documents
documents = load_faq_data()

# Combine question and answer for each document
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

# Embed in batches
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

# Search
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

  0%|          | 0/28 [00:00<?, ?it/s]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

To use a different model, add it to download.py, run the download, then update the path:

```python
embed = Embedder("models/Xenova/bge-base-en-v1.5")
vectors = embed.encode("your text here")
print(vectors.shape)
```